**Loading Llama3.2 model**

In [13]:
!pip install huggingface_hub
!mkdir -p models

from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id="hugging-quants/Llama-3.2-1B-Instruct-Q8_0-GGUF",
    filename="llama-3.2-1b-instruct-q8_0.gguf",
    local_dir="models"
)

'/content/models/llama-3.2-1b-instruct-q8_0.gguf'

In [14]:
!pip install langchain-community llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!pip install langchain
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="models/llama-3.2-1b-instruct-q8_0.gguf",
    n_gpu_layers=-1,
    n_ctx=2048,
    seed=42,
    verbose=False,
)

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121


Generating response for the prompt

In [15]:
def build_llama3_prompt(messages):
    prompt = "<|begin_of_text|>"
    for m in messages:
        prompt+= f"<|start_header_id|>{m['role']}<|end_header_id|>\n\n{m['content']}<|eot_id|>"
    prompt+= "<|start_header_id|>assistant<|end_header_id|>\n\n"
    return prompt


def generate(messages, max_new_tokens=200, do_sample=True, temperature=1, top_p=0.25):
    prompt = build_llama3_prompt(messages)
    return llm.invoke(
        prompt,
        max_tokens=max_new_tokens,
        temperature=temperature if do_sample else 0.0,
        top_p=top_p,
    )

In [16]:
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]
print(generate(messages))

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


Here's one:

Why did the chicken go to the doctor?

Because it had fowl breath! (get it?)


In [17]:
messages = [{"role": "user", "content": "Classify the text into neutral, negative or positive.\nText: I think the food was okay.\nSentiment:"}]
print(generate(messages))

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


The sentiment of the text is neutral. The word "okay" can have different connotations depending on the context, but in this case, it seems to be a neutral expression indicating that the food was not bad or excellent, just average.


In [18]:
persona = "You are an expert in AI programming assistant.Help solving, writing, explaining any code to make the user's work easy.\n"
instruction= "Give a step by step answer for the user's question.If it's a coding, answer should be working code.\n"
context = "The assistance is used by the some developers.\n"
data_format = """1. Question summery.
2. Explaination
3. Code (if applicabel)
4. Conclusion\n"""
audience = "The user is beginner to intermediate programer.\n"
tone = "Friendly, professional, and concise.\n"
data = "Explain me a C program for sum of two numbers in a simple way with code."
full_prompt = persona + instruction + context + data_format + audience + tone + data
messages = [{"role": "user", "content": full_prompt}]
print(generate(messages,max_new_tokens=350))

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


Here is a simple C program that calculates the sum of two numbers:

**sum_of_numbers.c**
```c
#include <stdio.h>

// Function to calculate the sum of two numbers
int sum(int num1, int num2) {
    return num1 + num2;
}

int main() {
    int num1 = 10; // First number
    int num2 = 20; // Second number

    printf("The sum of %d and %d is: %d\n", num1, num2, sum(num1, num2)));

    return 0;
}
```
**Explanation**

This C program calculates the sum of two numbers using a simple function `sum`. The function takes two integer arguments `num1` and `num2`, and returns their sum.

In the `main()` function, we declare two integer variables `num1` and `num2` with values 10 and 20 respectively. We then call the `sum()` function to calculate the sum of these two numbers, which is stored in the variable `result`.

Finally, we print the result to the console using `printf()`.


**Chain-of-Thought — zero-shot version**

In [19]:
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]
print(generate(zeroshot_cot_prompt, max_new_tokens=200))

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


Let's break down the problem step by step:

1. The cafeteria had 23 apples initially.
2. They used 20 apples to make lunch, leaving them with:
   - Initial number of apples: 23
   - Number of apples used for lunch: 20
   - Remaining apples after using some for lunch: 23 - 20 = 3

3. Then they bought 6 more apples.
4. Now, let's add the new apples to the remaining apples:
   - Initial number of apples: 23
   - Number of apples bought: 6
   - Total number of apples now: 23 + 6 = 29


**Tree-of-Thought**

In [20]:
zeroshot_tot_prompt = [
    {"role": "user", "content": (
        "Imagine three different experts are answering this question. "
        "All experts will write down 1 step of their thinking, then share it with the group. "
        "Then all experts will go on to the next step, etc. "
        "If any expert realizes they're wrong at any point then they leave. "
        "The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, "
        "how many apples do they have?' Make sure to discuss the results in short."
    )}
]
print(generate(zeroshot_tot_prompt, max_new_tokens=500))

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


Here's the step-by-step solution:

**Step 1:**
Expert 1 thinks: "The cafeteria had 23 apples."

**Step 2:**
Expert 2 thinks: "If they used 20 to make lunch, that means they have 3 left over from the original 23."

**Step 3:**
Expert 3 thinks: "But wait, if they bought 6 more apples, how many would they have in total?"

**Step 4:**
The experts discuss and arrive at a conclusion:

* Expert 1 is correct that there were 23 apples.
* Expert 2 is incorrect because the number of apples left over from lunch does not equal 3. In fact, it's likely to be much smaller than 3.
* Expert 3 is correct that if they bought 6 more apples, they would have a total of 29 apples.

The final answer is: $\boxed{29}$


**Output validation**

In [21]:
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:
{
  "description": "A SHORT DESCRIPTION",
  "name": "THE CHARACTER'S NAME",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [{"role": "user", "content": one_shot_template}]
output = generate(one_shot_prompt, max_new_tokens=150)
print(output)

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{
  "description": "A skilled warrior with a strong sense of justice.",
  "name": "Kaito Yamato",
  "armor": "Iron Gauntlets",
  "weapon": "Katana"
}


**Chain:**

*Multiple chaining*

In [36]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
title_prompt = PromptTemplate(template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>""",
    input_variables=["summary"]
)

title = title_prompt | llm



character_prompt = PromptTemplate(template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}.
Use only two sentences.<|end|>
<|assistant|>""",
    input_variables=["summary","title"]
)

character = character_prompt | llm



template = """<s><|user|>
Create a story about {summary} with the title {title}. The main character is:
{character}. Only return the story and it cannot be longer than one paragraph.
<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(template=template, input_variables=["summary", "title", "character"])
first_stage = RunnablePassthrough.assign(title=title)
second_stage = RunnablePassthrough.assign(character=character)
llm_chain = first_stage | second_stage | story_prompt | llm
result = llm_chain.invoke({
    "summary": "a girl that lost her mother"
})
print(result)

There is no user input. The story title is: "A Mother's Legacy".The main character is:
Sophia struggles to cope with the loss of her mother.
As she navigates her grief, she begins to uncover the secrets and stories that have been hidden in their family's past. Only return the story and it cannot be longer than one paragraph.
